# Experimentos Demo

## Piloto
Demonstração rápida com ações aleatórias e sem aprendizado

In [ ]:
from immune_rl.graph import build_pilot_graph
from immune_rl.dynamics import SimpleDynamics, PathogenInflammationReward
from immune_rl.environment import ImmuneEnvironment
from immune_rl.policies import RandomPolicy

In [2]:
env = ImmuneEnvironment(build_pilot_graph, SimpleDynamics(), PathogenInflammationReward())
policies = {c.id:RandomPolicy() for c in env.graph.cells()}

In [3]:
obs = env.reset()
done = False
while not done:
    actions = {cid: policies[cid].select_action(o) for cid, o in obs.items()}
    obs, rewards, terminated, truncated, info = env.step(actions)
    done = terminated or truncated
    print(f"t={info['t']:2d} {[a.name for a in actions.values()]}"
          f"P={info['P']:.2f} I={info['I']:.2f} R={rewards['mac_A']:+.3f}")


t= 1 ['REGULATE', 'ACTIVATE']P=1.00 I=0.06 R=+0.030
t= 2 ['ACTIVATE', 'ACTIVATE']P=0.90 I=0.17 R=+0.057
t= 3 ['NOOP', 'REGULATE']P=0.86 I=0.25 R=+0.036
t= 4 ['REGULATE', 'REGULATE']P=0.90 I=0.25 R=+0.003
t= 5 ['ACTIVATE', 'REGULATE']P=0.94 I=0.26 R=+0.002
t= 6 ['NOOP', 'REGULATE']P=0.99 I=0.26 R=+0.002
t= 7 ['NOOP', 'ACTIVATE']P=0.99 I=0.29 R=+0.017
t= 8 ['NOOP', 'NOOP']P=0.99 I=0.33 R=+0.015
t= 9 ['REGULATE', 'ACTIVATE']P=0.99 I=0.35 R=+0.014
t=10 ['ACTIVATE', 'NOOP']P=0.94 I=0.41 R=+0.027
t=11 ['NOOP', 'NOOP']P=0.89 I=0.46 R=+0.025
t=12 ['REGULATE', 'NOOP']P=0.89 I=0.47 R=+0.007
t=13 ['NOOP', 'ACTIVATE']P=0.85 I=0.51 R=+0.021
t=14 ['REGULATE', 'REGULATE']P=0.85 I=0.52 R=+0.004
t=15 ['REGULATE', 'ACTIVATE']P=0.81 I=0.56 R=+0.019
t=16 ['REGULATE', 'REGULATE']P=0.81 I=0.56 R=+0.002
t=17 ['NOOP', 'NOOP']P=0.81 I=0.57 R=+0.002
t=18 ['NOOP', 'REGULATE']P=0.85 I=0.54 R=-0.013
t=19 ['ACTIVATE', 'ACTIVATE']P=0.80 I=0.58 R=+0.018
t=20 ['NOOP', 'ACTIVATE']P=0.72 I=0.64 R=+0.031
t=21 ['REGULATE'

## Treinamento por reforço Multi-agente

Compara ações **aleatórias** x **Q-Learning independente** x **Q-Learning independente com política compartilhada**

In [4]:
import random
import numpy as np
from immune_rl.graph import build_pilot_graph
from immune_rl.dynamics import SimpleDynamics, PathogenInflammationReward
from immune_rl.environment import ImmuneEnvironment
from immune_rl.policies import RandomPolicy, QLearningPolicy
from immune_rl.encoding import StateEncoder

In [5]:
EPISODES = 1500

In [ ]:
def make_env():
    return ImmuneEnvironment(build_pilot_graph, SimpleDynamics(), PathogenInflammationReward())

def run_episode(env, policies, train):
    obs = env.reset()
    team_return = 0.0
    while True:
        actions = {cid: policies[cid].select_action(o) for cid, o in obs.items()}
        next_obs, rewards, terminated, truncated, info = env.step(actions)
        if train:
            for cid in obs:
                policies[cid].update(obs[cid], actions[cid], rewards[cid], next_obs[cid], terminated)
        team_return += sum(rewards.values()) / len(rewards)
        obs = next_obs
        if terminated or truncated:
            if train:
                for p in set(policies.values()):   # set: política compartilhada decai 1x só
                    p.end_episode()
            return team_return
 
 
def experiment(name, build_policies, seed):
    random.seed(seed)
    env = make_env()
    cells = [c.id for c in env.graph.cells()]
    policies = build_policies(cells, seed)
    returns = [run_episode(env, policies, train=True) for _ in range(EPISODES)]
    for p in set(policies.values()):
        if hasattr(p, "epsilon"):
            p.epsilon = 0.0                        # avaliação de exploit
    evals = [run_episode(env, policies, train=False) for _ in range(200)]
    return np.mean(returns[:100]), np.mean(returns[-100:]), np.mean(evals)
 
 
def random_pols(cells, seed):
    return {c: RandomPolicy() for c in cells}
 
def independent(cells, seed):
    return {c: QLearningPolicy(StateEncoder(), seed=seed + i) for i, c in enumerate(cells)}
 
def shared(cells, seed):
    p = QLearningPolicy(StateEncoder(), seed=seed)
    return {c: p for c in cells}